1. breast_cancer 로드 + RandomForest/XGBoost 학습·평가 (10분)
2. entropy 함수 구현 + 순수 노드에서 0이 나오는지 확인 (5분)
3. information_gain 함수 구현 (5분)
4. best_split으로 특정 데이터의 최적 분할점 찾기 + sklearn과 비교 (15분)
5. CLT 시뮬레이션: 지수분포에서 표본평균 분포가 정규분포에 가까워지는지 확인 (15분)

30분 이상 막히면 Day1~4 코드 참고 가능.
목표는 코드를 외우는 게 아니라 "불확실성(엔트로피) → 분할 기준 →
앙상블"이라는 흐름과, CLT가 왜 통계 전반의 기반이 되는지를 손에 익히는 것.

In [35]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import numpy as np

from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score

In [36]:
data = load_breast_cancer()
def model_eval(X: np.ndarray, y:np.ndarray):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    model_a = RandomForestClassifier()
    model_b = XGBClassifier()
    models = [model_a, model_b]
    results = {}
    for model in models:
        model.fit(X_train, y_train)
        roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1],multi_class='ovr')
        acc = accuracy_score(y_test, model.predict(X_test))
        results[model] = {'acc' : acc, 'roc_acu' : roc_auc}
    return results

model_eval(data.data, data.target)

{RandomForestClassifier(): {'acc': 0.9649122807017544,
  'roc_acu': 0.9959582598471488},
 XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=True, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=None, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
               max_leaves=None, min_child_weight=None, missing=nan,
               monotone_constraints=None, multi_strategy=None, n_estimators=None,
               n_jobs=None, num_parallel_tree=None, ...): {'acc': 0.9707602339181286,
  'roc_acu': 0.9951499118165784}}

In [37]:
def entropy(X):
    values, counts = np.unique(X,return_counts=True)
    result = 0
    for count in counts:
        result = -count/sum(counts)*np.log2(count/sum(counts))
    return result
entropy([1,1,1,1])
entropy(data.target)

np.float64(0.4219404823287542)

In [38]:
def information_gain(y_parent, y_left, y_right):
    p_left = len(y_left)/len(y_parent)
    p_right = len(y_right)/len(y_parent)
    return entropy(y_parent)-p_left*entropy(y_left)-p_right*entropy(y_right)
feature_idx = data.feature_names.tolist().index("mean concave points")
feature_values = np.unique(feature_idx)

y= data.target
x = data.data[:,feature_idx]
y_left   = y[x >= 0.01]           
y_right  = y[x < 0.01]    
information_gain(y,y_left,y_right)   

np.float64(0.013238837201822118)

In [41]:
def best_split(x, y):
    #중간값 생성 함수
    def mid(values):
        return (values[:len(values)-1] + values[1:])/2
    #특성마다 중간값 생성
    mid_points = []
    
    
    for i in range(x.shape[1]):
        values, counts = np.unique(x[:,i], return_counts=True)
        mid_points.append(mid(values))
    print(len(mid_points))
    #특성마다 ig비교
    idxs = []
    result = []
    for mid_point in range(len(mid_points)):
        igs = []
        for i in range(len(mid_points[mid_point])):
            y_left = y[x[:,mid_point]<=mid_points[mid_point][i]]
            y_right = y[x[:,mid_point]>mid_points[mid_point][i]]
            igs.append(information_gain(y,y_left,y_right))
            
        idx = np.argmax(igs)
        idxs.append(mid_points[mid_point][idx])
        result.append(igs[idx]) 
    idx = np.argmax(result)

    return idx, result[idx]

best_split(data.data, data.target)


30


(np.int64(22), np.float64(0.2875888868203324))